# ✅ Preguntas fáciles

1. **¿Qué es un socket y para qué sirve en tu servidor?**
son nodos o puntos en la red, que permiten intercambias informacoin entre servidores y clientes. En mi servidor, cada cliente se conecta a traves de un socket para eenviar y recibir mensajes

2. **¿Qué diferencia hay entre bind() y listen()?**
bind - asocia una direccion ip y puerto a un socket
listen - pone al servidor en modo escucha, para ver si hay alguien intentando hacer una conexion

3. **¿Qué hace accept() dentro del servidor?**
accept - acepta las conexiones entrantes, cuanddo se acepta la conexion devuelve un nuevo socket para comunicarse con ese cliente y la direccion del mismo

4. **¿Por qué usás un while True dentro del servidor?**
Por que el servidor debe estar escuchando continuamente mientras no este caido, para poder aceptar nuevas conexiones y manejar los mensajes

5. **¿Qué representa SERVER_HOST y SERVER_PORT?**
SERVER_HOST ip local del servidor
SERVER_PORT puerto que se utilizara para escuchar las conexiones entrantes

6. **¿Para qué usás threading.Thread en tu servidor?**
para que cada cliente tenga su propio hilo de ejecucion

7. **¿Por qué cada cliente necesita su propio hilo?**
Por que si un cliente se queda esperando datos o se bloquea, podria afectar el funcionamiento en general, con un hilo por cliente, cada uno se maneja de forma independiente

8. **¿Qué pasa cuando un cliente se desconecta abruptamente?**
El servidor recibe un error o un mensaje vacio, el servidor debe cerrar el socket del cliente, sin afectar el resto del servidor

9. **¿Para qué sirve un try/except en la parte donde recibís mensajes?**
Manejo de errores, para evitar que un error al recibir datos(por desconexion o fallo del cliente) haga que el servidor se caiga. Permite manejar errores por cliente de forma controlada

10. **¿Qué hace la función que maneja la recepción de mensajes de un cliente?**
Recibe los datos que el cliente envia, y envia a los demas clientes(broadcast), tambien detecta cuando el cliente se desconecta(por que se recibe un mensaje vacio)

# 🔶 Preguntas de dificultad media 

11. **¿Qué diferencia hay entre usar hilos daemon y no daemon en un servidor TCP?**
    Hilos daemon - se cierran automaticmente cuando el programa principal termina, No garantiza que el codigo del hilo termine correctamente
    Hilos no daemon - continuan ejecutandose hasta finalizar, aunque el programa principal termine

    **En servidores, normalmente los hilos NO deberian ser daemon, por que necesitan limpiar recursos correctamente**


12. **Si un cliente se desconecta, ¿por qué tu servidor muestra el mensaje correctamente pero los demás clientes dejan de enviar mensajes?**
Por que probablemente el hilo que maneja ese cliente, lanza una excepcion que no esta correctamente aislada, o la excepcion rompe un bucle global

**Una excepcion no controlada en un hilo puede bloquear el funcionamiento del servidor**

13. **¿Qué sucede dentro del hilo cuando recv() devuelve 0 bytes?**
Significa que el cliente cerro la conexion
Python interpreta un recv() vacio como un fin de la conexion, lo correcto es cerrar el socket y terminar el hilo 

14. **¿Por qué es importante manejar excepciones por cliente y no de forma global?**
Porque si el servidor usa un único try/except para todo, entonces un error de un cliente podría detener todo el servidor o bloquear otros hilos.

Cada hilo debe manejar sus propios errores para evitar afectar a otros clientes.

15. **¿Qué riesgo existe si usás un solo try/except alrededor del bucle principal del servidor?**
Que una unica excepcion detenga:

- el bucle de aceptación,

- todos los hilos activos,

- y deje el servidor inutilizable.

El servidor debe tener try/except independientes en cada sección crítica.

16. **¿Qué pasaría si accept() se ejecutara sin estar dentro de un hilo independiente?**
El servidor quedaría bloqueado esperando conexiones nuevas y no podria hacer ninguna otra tarea mientras tanto.

Ejemplo de problemas:

- no podría cerrar correctamente,

- no podría procesar señales,

- no podría manejar otras funciones paralelas.

17. **¿Qué problema puede causar usar sock.recv(1024) sin validar el contenido?**
Puede recibir:

- mensajes vacíos (desconexión),

- datos corruptos,

- datos incompletos de un mensaje grande,

- mensajes binarios inesperados.

18. **¿Cómo afecta el uso de variables compartidas entre hilos si no usás locks?**
Puede generar condiciones de carrera (race conditions).

Ejemplos:

- dos hilos editan la lista de clientes al mismo tiempo → se corrompe,

- eliminación duplicada de un socket,

- mensajes enviados a clientes ya desconectados.

Los locks evitan que varios hilos modifiquen un recurso al mismo tiempo.

19. **¿Cómo podrías evitar que un error en un cliente afecte a todos los demás?**
Aislando cada cliente en su propio hilo y usando un try/except interno.
Además, evitando usar estructuras compartidas sin control, y asegurando que cada desconexión se maneje limpiamente sin modificar variables globales de forma insegura.

20. **¿Qué pasaría si el servidor no cierra correctamente los sockets al terminar?**
Los sockets pueden quedar:

- abiertos en el sistema operativo,

- en estado TIME_WAIT, bloqueando el puerto,

- consumiendo recursos innecesarios.

Esto podría impedir que el servidor se reinicie inmediatamente o provocar fugas de recursos.


# 🔥 Preguntas difíciles

21. **¿Por qué un hilo que maneja un cliente desconectado podría bloquear el servidor completo si no está correctamente aislado?**
Porque si ese hilo ejecuta una operación bloqueante (por ejemplo recv() sin timeout, send() a un socket que no acepta escritura, o espera por un lock compartido) y además comparte recursos críticos con otras partes del servidor, entonces:

si el hilo toma un lock que otros necesitan y queda bloqueado, esos otros hilos también quedan bloqueados -> deadlock o bloqueo en cascada;

si el hilo ejecuta recv() sin timeout dentro del hilo principal (o en un hilo que debe atender otras tareas), impide que avance la lógica de aceptación o cierre;

si el hilo provoca una excepción no capturada y ese error se propaga de forma global (p. ej. rompe el bucle principal si el try/except es limitado), puede detener la aceptación de nuevas conexiones.

22. **Explicá por qué un servidor con threads daemon puede finalizar sin cerrar conexiones activas.**
Un hilo marcado como daemon le dice al intérprete Python: "no esperes a que este hilo termine al cerrar el proceso" — cuando el proceso principal termina, todos los hilos daemon se terminan abruptamente (el runtime no llama a su finally, no ejecuta join(), etc.). Eso causa que:

buffers de envío no se vacíen (mensajes pendientes se pierden);

no se ejecuten rutinas de limpieza (cerrar sockets con shutdown()/close()), dejando recursos o descriptors abiertos;

los clientes ven que la conexión se corta de golpe (RST) o quedan en estados inconsistentes.

Regla práctica: usar hilos no-daemon para trabajo que requiere limpieza y usar join() y eventos para cerrar ordenadamente. Usa daemon solo para threads auxiliares cuyo trabajo no requiere limpieza obligatoria.

23. **¿Cuál es la diferencia entre un servidor multihilo y un servidor basado en select() o asyncio?**
**Servidor multihilo:**

1 hilo por conexión (tradicional).

Fácil de razonar (cada hilo tiene su stack y flujo).

Mala escalabilidad: cada hilo consume memoria y CPU por cambio de contexto; 10k hilos suele ser impracticable.

Buena para tareas que hacen CPU o bloqueo puntual si hay pocos clientes.

**Servidor select() / event-driven / asyncio:**

Un hilo (o un pool pequeño) maneja muchas conexiones usando notificaciones de I/O (select/poll/epoll/kqueue).

Escala a miles de conexiones con baja memoria por conexión.

Requiere programar de forma no bloqueante (callbacks, async/await).

Mejor rendimiento en I/O-bound y alta concurrencia.

**Cuándo elegir:**

<100-500 conexiones y mucha lógica por conexión -> multihilo es sencillo.

Miles de conexiones simultáneas, I/O-bound, necesidad de eficiencia -> usar asyncio / epoll / event loop.

24. **¿Cómo evitarías condiciones de carrera si varios hilos modifican una lista de clientes?**
Usando mecanismos de sincronización o evitando accesos concurrentes

25. **¿Cómo implementarías un mecanismo seguro para difundir mensajes a todos los clientes activos?**
Patrón recomendado:

1. Mantener por cliente una cola de envío (queue.Queue) thread-safe.

2. El thread/loop que produce (ej. handle_client) encola el mensaje en la cola de cada target (sin I/O).

3. Cada cliente tiene un sender thread que solo saca mensajes de su cola y hace sendall.

4. Para broadcast, copiar la lista de colas bajo lock y luego encolar sin lock.

**Ventajas:**

Evita que un send lento bloquee el thread que hace broadcast.

Puedes imponer límites a cada cola (backpressure) y desconectar clientes que no consumen.

**Ejemplo de broadcast seguro (simplificado):**
            
            with clients_lock:
                queues = [info["queue"] for info in clients.values()]
            for q in queues:
                try:
                    q.put_nowait(msg)
                except queue.Full:
                    # aplicar política: dropear, desconectar, o bloquear breve
                    handle_slow_client(q)


26. **¿Cómo detectarías correctamente la desconexión silenciosa de un cliente (sin enviar FIN)?**
Las desconexiones silenciosas (cuando el cliente se cae sin hacer close()) son difíciles porque TCP no siempre informa inmediatamente. Tácticas:

Keepalive TCP: habilitar TCP keepalive a nivel OS/socket para detectar enlaces muertos (configurable intervalos); detecta desconexión por falta de ACK en bajo nivel.

En Python: sock.setsockopt(socket.SOL_SOCKET, socket.SO_KEEPALIVE, 1) y en sistemas ajustar parámetros del SO (intervalos).

Heartbeats a nivel de aplicación: protocolo simple donde cliente envía (o servidor solicita) ping cada N segundos y espera pong. Si no recibe respuesta en M intentos, considera desconectado.

Ej: cada cliente debe enviar un ping cada 30s; si 3 pings fallan -> desconexión.

Timeouts en recv(): usar timeouts y contar tiempo desde la última actividad. Si excede X segundos sin datos, marcarlo como inactivo y cerrar.

Combinar: keepalive + heartbeats + timeouts robustecen la detección.

27. **¿Qué estrategia usarías para permitir que el servidor cierre con Ctrl + C sin perder mensajes ni corromper sockets?**
Procedimiento de apagado ordenado:

1. Capturar KeyboardInterrupt (o señal SIGINT) en el hilo principal y marcar un Event global shutting_down.set().

2. Dejar de aceptar nuevas conexiones (cerrar server_sock o salir del loop de accept).

3. Notificar a los clientes: encolar un mensaje de cierre o "server shutting down" en cada cola para que los senders lo despachen.

4. Señalizar senders: poner None o mensaje especial en colas para que los sender threads terminen tras vaciar la cola.

5. Cerrar sockets ordenadamente: llamar shutdown() luego close() para cada socket.

6. Esperar hilos: join() con timeout razonable para cada thread (recvers y senders).

7. Forzar cierre si excede timeout: si algún hilo no responde, como último recurso cerrar sockets y terminar.

**Esto asegura que:**

- los mensajes en cola son enviados antes de cerrar (si se puede),

- no hay descriptors huérfanos,

- el cierre no interrumpe I/O a medias sin señalización.

